##### 常數

In [11]:
CAMERA_ID = 0
MODEL = "./models/BlazePose/pose_landmarker_full.task"


#### 套件

In [12]:
import cv2
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg

from mediapipe.tasks.python.core.base_options import BaseOptions
from mediapipe.tasks.python.vision.core.vision_task_running_mode import (
    VisionTaskRunningMode,
)
from mediapipe.tasks.python.vision.pose_landmarker import (
    PoseLandmarker,
    PoseLandmarkerOptions,
    PoseLandmarkerResult,
)

import mediapipe as mp
import numpy as np
import time

from mpl_toolkits.mplot3d.axes3d import Axes3D

In [13]:
from src.calculate import (
    draw_circles_by_landmarks,
    draw_bones_in_plot_by_pose_lanmark_result,
)
from src.utils.plot_painter import set_data_range, set_plot_labels
from src.utils.cv_lib import draw_letter_badge
from src.mediapipe_lib.base import PoseLandmarkerLiveStream, PoseResult, ResultAnalyzer


#### 忽略警告

In [14]:
import warnings

warnings.filterwarnings("ignore")

#### 函式

##### 在原始圖片上繪製姿勢結果

In [15]:
def draw_result_in_img(
    result: PoseLandmarkerResult,
    origin_img: cv2.typing.MatLike,
) -> cv2.typing.MatLike:
    """在原始圖片上繪製姿勢結果

    Args:
        result (PoseLandmarkerResult): 姿勢結果
        origin_img (cv2.typing.MatLike): 原始圖片

    Returns:
        cv2.typing.MatLike: 繪製過姿勢結果的圖片
    """
    IS_3D = True  # 是否為 3D 姿勢

    # 複製原始圖片
    result_img = origin_img.copy()

    # 未取得任何運算結果
    if len(result.pose_landmarks) <= 0:
        result_img = draw_letter_badge(result_img, "F", (30, 230))
        return result_img

    # 建立分析器
    pose_result = PoseResult(result)
    analyzer = ResultAnalyzer(pose_result)

    ### 繪製關鍵點
    result_img = draw_circles_by_landmarks(result_img, result.pose_landmarks)

    ### 繪製驗證標誌
    # 軀幹是否扭轉
    if analyzer.check_if_trunk_is_twisted(IS_3D):
        result_img = draw_letter_badge(result_img, "A", (30, 30))
    # 手是否遠離身體
    if analyzer.check_if_hands_at_a_distance(IS_3D):
        result_img = draw_letter_badge(result_img, "B", (30, 80))
    # 手臂是否抬起，水平位置且位於肩膀和手肘間
    if analyzer.check_if_arms_raised(IS_3D):
        result_img = draw_letter_badge(result_img, "C", (30, 130))
    # 手是否高過肩膀
    if analyzer.check_if_hands_above_shoulder(IS_3D):
        result_img = draw_letter_badge(result_img, "D", (30, 180))

    return result_img

##### 取得姿勢表格座標

In [16]:
def get_pose_plot_axes(result: PoseLandmarkerResult, ax: Axes3D = None):
    """取得姿勢表格座標

    Args:
        result (PoseLandmarkerResult): 姿勢分析結果
        ax (Axes3D, optional): 3D 表格座標. Defaults to None.

    Returns:
        Axes3D: 表格座標
    """
    DATA_RANGE = ([-1, 1], [-1, 1], [0, 2])  # 顯示資料範圍
    VIEW_INIT = (30, -60, 0)  # 初始視角
    IS_3D = True  # 是否為 3D 姿勢

    if ax is None:
        plt.gca()

    set_data_range(DATA_RANGE[0], DATA_RANGE[1], DATA_RANGE[2], ax)
    ax.view_init(VIEW_INIT[0], VIEW_INIT[1], VIEW_INIT[2])
    set_plot_labels("x", "z", "y", ax)

    if len(result.pose_landmarks) > 0:
        pose_result = PoseResult(result)
        analyzer = ResultAnalyzer(pose_result)

        ax.set_title(f"Label: {analyzer.get_lhc_label(IS_3D)}")
        draw_bones_in_plot_by_pose_lanmark_result(pose_lanmark_result=result, ax=ax)
        ax.legend()
    else:
        ax.set_title("NO DATA")

    return ax

##### 取得肩臀交錯角度圖表座標

In [17]:
def get_staggered_angle_axes(result: PoseLandmarkerResult, ax: Axes3D = None) -> Axes3D:
    """取得肩臀交錯角度圖表座標

    Args:
        result (PoseLandmarkerResult): 姿勢分析結果
        ax (Axes3D, optional): 3D 圖表座標. Defaults to None.

    Returns:
        Axes3D: 圖表座標
    """
    DATA_RANGE = ([-1, 1], [-1, 1], [1, -1])  # 顯示資料範圍
    VIEW_INIT = (90, 0, 0)  # 初始視角
    IS_3D = True  # 是否為 3D 姿勢

    # 取得當前圖表座標
    if ax is None:
        plt.gca()

    # 設定圖表資料
    set_data_range(DATA_RANGE[0], DATA_RANGE[1], DATA_RANGE[2], ax)
    ax.view_init(VIEW_INIT[0], VIEW_INIT[1], VIEW_INIT[2])
    set_plot_labels("x", "z", "y", ax)

    # 繪製圖表資訊
    if len(result.pose_landmarks) > 0:
        pose_result = PoseResult(result)
        analyzer = ResultAnalyzer(pose_result)

        # 繪製肩臀線條
        shoulder = np.array(
            [
                pose_result.get_kpt_pos_by_name("left_shoulder", IS_3D),
                pose_result.get_kpt_pos_by_name("right_shoulder", IS_3D),
            ]
        )
        hip = np.array(
            [
                pose_result.get_kpt_pos_by_name("left_hip", IS_3D),
                pose_result.get_kpt_pos_by_name("right_hip", IS_3D),
            ]
        )
        ax.set_title(
            f"Staggered Angle: {analyzer.get_pose_shoulder_hip_staggered_angle(IS_3D):.2f}"
        )
        ax.plot(shoulder[:, 0], shoulder[:, 2], shoulder[:, 1], label="shoulder")
        ax.plot(hip[:, 0], hip[:, 2], hip[:, 1], label="hip")
        ax.legend()
    else:
        ax.set_title("NO DATA")

    return ax

##### 取得雙手中心和身體重心資訊的圖表座標

In [18]:
def get_hand_to_body_gravity_axes(
    result: PoseLandmarkerResult, ax: Axes3D = None
) -> Axes3D:
    """取得雙手中心和身體重心資訊的圖表座標

    Args:
        result (PoseLandmarkerResult): 姿勢分析結果
        ax (Axes3D, optional): 3D 圖表座標. Defaults to None.

    Returns:
        Axes3D: 圖表座標
    """
    DATA_RANGE = ([-1, 1], [-1, 1], [0, 2])  # 顯示資料範圍
    VIEW_INIT = (30, -120, 0)  # 初始視角
    IS_3D = True  # 是否為 3D 姿勢

    # 取得當前圖表座標
    if ax is None:
        plt.gca()

    # 設定圖表資料
    set_data_range(DATA_RANGE[0], DATA_RANGE[1], DATA_RANGE[2], ax)
    ax.view_init(VIEW_INIT[0], VIEW_INIT[1], VIEW_INIT[2])
    set_plot_labels("x", "z", "y", ax)

    # 繪製圖表資訊
    if len(result.pose_landmarks) > 0:
        pose_result = PoseResult(result)
        analyzer = ResultAnalyzer(pose_result)

        # 顯示雙手中心和身體重心距離
        ax.set_title(
            f"distance: {analyzer.get_hand_and_body_gravity_distance(IS_3D):.2f}"
        )
        draw_bones_in_plot_by_pose_lanmark_result(pose_lanmark_result=result, ax=ax)

        # 取得全關鍵點中的 y 軸資訊
        all_kpts = np.array(pose_result.get_all_kpt_positions(IS_3D))
        kpt_y = all_kpts[:, 1] * -1
        y_min = kpt_y.min()  # 取得全關鍵點最小的 y 值

        # 取得 hand_to_gravity 座標
        hand_to_gravity = np.array(
            [
                analyzer.get_center_position_by_2_hand(IS_3D),
                analyzer.get_body_gravity_position(IS_3D),
            ]
        )

        # 轉換 hand_to_gravity y 軸座標
        hand_to_gravity[:, 1] *= -1
        hand_to_gravity[:, 1] -= y_min

        ax.plot(
            hand_to_gravity[:, 0],
            hand_to_gravity[:, 2],
            hand_to_gravity[:, 1],
            c="#f0f",
            label="hand to body gravity line",
        )
        ax.scatter(
            hand_to_gravity[0, 0],
            hand_to_gravity[0, 2],
            hand_to_gravity[0, 1],
            c="#880",
            label="hand center",
        )
        ax.scatter(
            hand_to_gravity[1, 0],
            hand_to_gravity[1, 2],
            hand_to_gravity[1, 1],
            c="#808",
            label="body gravity",
        )

        ax.legend()
    else:
        ax.set_title("NO DATA")

    return ax

##### 繪製分析結果圖表

In [19]:
def draw_result_plot(
    result: PoseLandmarkerResult,
    figsize: tuple[float, float] = (6.4, 4.8),
) -> cv2.typing.MatLike:
    """繪製分析結果圖表

    Args:
        result (PoseLandmarkerResult): 姿勢結果
        figsize (tuple[float, float], optional): 圖表大小. Defaults to (6.4, 4.8).

    Returns:
        cv2.typing.MatLike: 分析結果圖片
    """
    ### 繪製表格
    fig = plt.figure(figsize=figsize)

    # 建立座標
    ax: Axes3D = fig.add_subplot(projection="3d")
    get_hand_to_body_gravity_axes(result, ax)

    # # 繪製座標 1
    # ax_0: Axes3D = fig.add_subplot(121, projection="3d")
    # get_pose_plot_axes(result, ax_0)
    # # 繪製座標 2
    # ax_1: Axes3D = fig.add_subplot(122, projection="3d")
    # get_hand_to_body_gravity_axes(result, ax_1)

    # 固定圖表內容
    canvas = FigureCanvasAgg(fig)
    canvas.draw()
    plt.close(fig)

    # 將圖表 buffer 轉換成 numpy
    rgba = np.asarray(canvas.buffer_rgba())
    rgb = cv2.cvtColor(rgba, cv2.COLOR_RGBA2RGB)

    return rgb

#### 開始測試

In [20]:
cap = cv2.VideoCapture(CAMERA_ID)  # 取得串流資訊
live = PoseLandmarkerLiveStream(MODEL)  # 建立模型串流物件

# 串流大小(寬高)
cap_size = (
    int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
    int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, cap_size[0])
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, cap_size[1])
# 輸出表格大小
plot_size = (cap_size[0] / 100, cap_size[1] / 100)

# 預設輸出影像畫面
default_out_frame = np.zeros((cap_size[1], cap_size[0], 3), np.uint8)

# 進行串流
while cap.isOpened():
    # 抓取影像
    ret, frame = cap.read()
    if not ret:
        print("Can't receive frame (stream end?). Exiting ...")

    # 進行影像分析
    live.detect_async(frame, int(time.time() * 1000))
    mp_img = live.current_image
    result = live.result
    timestamp = live.current_timestamp_ms

    # 模型是否取得影像畫面
    if mp_img is None:
        mp_img = default_out_frame.copy()
    else:
        mp_img = np.array(mp_img.numpy_view())

    ### 繪製分析結果
    mp_img = draw_result_in_img(result, mp_img)  # 繪製 2D 關鍵點
    plot_img = draw_result_plot(result)  # 繪製分析圖表
    # 組合原始圖片和分析圖表
    out_frame = np.concatenate((mp_img, plot_img), axis=1)

    # 顯示輸出影像
    cv2.imshow("debug", out_frame)
    if cv2.waitKey(1) == ord("q"):
        break


live.close()
cap.release()
cv2.destroyAllWindows()
print("finish")

finish
